# Fraud Transaction Intent Classification — Dataset Overview

**Domain:** Finance
**Problem Type:** Binary Classification
**Total Rows:** 15,000
**Total Columns:** 18

---

## About the Dataset

This dataset contains financial transaction records labeled for fraud detection. Each row represents a single transaction captured across various behavioral, geographic, device, and authentication-related attributes. The goal is to classify whether a given transaction is fraudulent or legitimate based on the available features.

---

## Column Descriptions

| # | Column | Description |
|---|---|---|
| 1 | `transaction_id` | Unique identifier for each transaction |
| 2 | `transaction_amount` | The monetary value of the transaction in USD |
| 3 | `transaction_hour` | The hour of the day (0–23) when the transaction was made |
| 4 | `device_type` | The type of device used to initiate the transaction |
| 5 | `location_mismatch` | Indicates whether the transaction location differs from the cardholder's registered address |
| 6 | `previous_fraud_flags` | The number of past fraud flags associated with the account |
| 7 | `velocity_last_1hr` | The number of transactions made by the user in the past hour |
| 8 | `merchant_category` | The category of the merchant involved in the transaction |
| 9 | `card_type` | The type of payment card used |
| 10 | `account_age_days` | The age of the account in days at the time of the transaction |
| 11 | `failed_auth_attempts` | The number of failed authentication attempts prior to this transaction |
| 12 | `ip_country_match` | Whether the IP address country matches the card-issuing country |
| 13 | `amount_deviation_ratio` | How much the transaction amount deviates from the user's historical average |
| 14 | `transaction_channel` | The channel through which the transaction was conducted |
| 15 | `is_weekend` | Whether the transaction took place on a weekend |
| 16 | `distance_from_home_km` | The distance in kilometers between the transaction location and the user's home |
| 17 | `cvv_match` | Whether the CVV provided matched the card's CVV |
| 18 | `is_fraudulent_transaction` | **Target** — Whether the transaction is fraudulent (1) or legitimate (0) |


In [28]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import gaussian_kde
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

import joblib



# Data Cleaning

In [29]:
df = pd.read_csv('fraud_transaction_dataset.csv')
# Remove the 'transaction_id' column
df.drop(columns=['transaction_id'], inplace=True)

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_amount         11332 non-null  float64
 1   transaction_hour           13622 non-null  float64
 2   device_type                13573 non-null  object 
 3   location_mismatch          13597 non-null  float64
 4   previous_fraud_flags       13580 non-null  float64
 5   velocity_last_1hr          11395 non-null  float64
 6   merchant_category          13557 non-null  object 
 7   card_type                  13675 non-null  object 
 8   account_age_days           11383 non-null  float64
 9   failed_auth_attempts       11385 non-null  float64
 10  ip_country_match           13602 non-null  float64
 11  amount_deviation_ratio     11375 non-null  float64
 12  transaction_channel        13603 non-null  object 
 13  is_weekend                 13549 non-null  flo

In [31]:
df.head(10)

,transaction_amount,transaction_hour,device_type,location_mismatch,previous_fraud_flags,velocity_last_1hr,merchant_category,card_type,account_age_days,failed_auth_attempts,ip_country_match,amount_deviation_ratio,transaction_channel,is_weekend,distance_from_home_km,cvv_match,is_fraudulent_transaction
0,NaN,9.0,desktop,0.0,0.0,0.0,travel,NaN,170.0,0.0,1.0,-0.2920,NaN,0.0,5.89,NaN,0
1,NaN,NaN,mobile,0.0,0.0,NaN,travel,credit,189.0,0.0,1.0,NaN,pos,1.0,7.63,NaN,0
2,172.55,15.0,desktop,0.0,0.0,NaN,groceries,debit,NaN,0.0,1.0,-0.7713,mobile_app,0.0,22.43,1.0,0
3,2729.47,21.0,desktop,0.0,0.0,9.0,gambling,prepaid,22.0,4.0,NaN,5.9249,online,0.0,202.49,NaN,1
4,232.76,NaN,mobile,1.0,1.0,0.0,entertainment,debit,557.0,1.0,1.0,-0.5409,online,0.0,NaN,1.0,0
5,NaN,13.0,mobile,0.0,0.0,0.0,electronics,virtual,613.0,0.0,1.0,NaN,online,0.0,NaN,1.0,0
6,NaN,15.0,desktop,NaN,0.0,2.0,groceries,NaN,159.0,0.0,1.0,NaN,NaN,0.0,6.49,1.0,0
7,101.88,11.0,mobile,1.0,0.0,NaN,groceries,NaN,241.0,1.0,1.0,NaN,online,1.0,NaN,1.0,0
8,214.54,NaN,mobile,0.0,0.0,NaN,groceries,NaN,1080.0,0.0,1.0,NaN,pos,1.0,0.08,1.0,0
9,192.56,8.0,mobile,0.0,0.0,NaN,retail,credit,2610.0,0.0,1.0,-0.3788,online,0.0,0.99,1.0,0


# Description Analysis
- Sqewness in transaction_amount and amount_deviation_ratio since the mean and median values differ by a large amount for it to be sqewed. Prefer to use median instead of mean

In [32]:
df.describe()
#

,transaction_amount,transaction_hour,location_mismatch,previous_fraud_flags,velocity_last_1hr,account_age_days,failed_auth_attempts,ip_country_match,amount_deviation_ratio,is_weekend,distance_from_home_km,cvv_match,is_fraudulent_transaction
count,11332.000000,13622.000000,13597.000000,13580.000000,11395.000000,11383.000000,11385.000000,13602.000000,11375.000000,13549.000000,11439.000000,11325.000000,15000.000000
mean,20898.843480,12.197989,0.104214,0.284315,1.661167,743.224458,0.294071,0.889796,109.745952,0.292199,50.775382,0.945430,0.080000
std,113252.389252,5.277158,0.305549,0.817154,2.157905,785.274380,0.901288,0.313156,590.604950,0.454790,173.497892,0.227148,0.271302
min,5.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,-0.994600,0.000000,0.000000,0.000000,0.000000
25%,52.235000,8.000000,0.000000,0.000000,0.000000,176.000000,0.000000,1.000000,-0.724300,0.000000,6.190000,1.000000,0.000000
50%,126.260000,12.000000,0.000000,0.000000,1.000000,500.000000,0.000000,1.000000,-0.183200,0.000000,14.910000,1.000000,0.000000
75%,277.700000,16.000000,0.000000,0.000000,2.000000,1026.000000,0.000000,1.000000,1.439300,1.000000,31.690000,1.000000,0.000000
max,995267.410000,23.000000,1.000000,9.000000,17.000000,8786.000000,10.000000,1.000000,4988.200200,1.000000,3438.970000,1.000000,1.000000


In [33]:
# Check to see if our target variable is imbalanced
fraud_counts = df['is_fraudulent_transaction'].value_counts()
print(fraud_counts)
# Imblanced target variable, so we will need to address this before modeling. 
# We can use SMOTE to oversample the minority class in the training set after splitting the data.

is_fraudulent_transaction
0    13800
1     1200
Name: count, dtype: int64


# Exploratory Data Analysis

In [34]:
eda = df.copy()
eda['Fraud'] = eda['is_fraudulent_transaction'].map({0: 'Legit', 1: 'Fraud'})

# 1. Strip — Transaction Amount by Fraud label (capped at 99th pct)
cap  = eda['transaction_amount'].quantile(0.99)
fig1 = px.strip(
    eda[eda['transaction_amount'] <= cap],
    x='Fraud', y='transaction_amount',
    color='Fraud',
    color_discrete_map={'Legit': '#1f77b4', 'Fraud': '#d62728'},
    title='Transaction Amount by Fraud Label (capped at 99th percentile)',
    labels={'transaction_amount': 'Transaction Amount (USD)', 'Fraud': ''},
    template='plotly_white'
)
fig1.update_traces(jitter=0.4, marker=dict(size=3, opacity=0.4))
fig1.show()

# 2. Box — Distance from Home by Fraud label
eda['distance_log'] = np.log1p(eda['distance_from_home_km'])

fig2 = px.box(
    eda.dropna(subset=['distance_log']),
    x='Fraud', y='distance_log',
    color='Fraud',
    color_discrete_map={'Legit': '#1f77b4', 'Fraud': '#d62728'},
    title='Distance from Home by Fraud Label (log scale)',
    labels={'distance_log': 'log(Distance from Home + 1 km)', 'Fraud': ''},
    points='outliers',
    template='plotly_white'
)
fig2.show()

# 3. Grouped Bar — Weekend vs Weekday fraud rate (normalised to % within each day type)
weekend_grp = (
    eda.dropna(subset=['is_weekend'])
       .groupby(['is_weekend', 'Fraud'])
       .size().reset_index(name='count')
)
weekend_grp['is_weekend'] = weekend_grp['is_weekend'].map({0.0: 'Weekday', 1.0: 'Weekend'})
totals = weekend_grp.groupby('is_weekend')['count'].transform('sum')
weekend_grp['pct'] = (weekend_grp['count'] / totals * 100).round(2)

fig3 = px.bar(
    weekend_grp, x='is_weekend', y='pct', color='Fraud',
    barmode='group',
    color_discrete_map={'Legit': '#1f77b4', 'Fraud': '#d62728'},
    title='Fraud vs Legitimate Transactions: Weekday vs Weekend (% of total)',
    labels={'is_weekend': '', 'pct': '% of Transactions'},
    text='pct',
    template='plotly_white'
)
fig3.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig3.update_layout(yaxis_ticksuffix='%')
fig3.show()

# 4. Box — Failed Auth Attempts by Fraud label
fig4 = px.box(
    eda.dropna(subset=['failed_auth_attempts']),
    x='Fraud', y='failed_auth_attempts',
    color='Fraud',
    color_discrete_map={'Legit': '#1f77b4', 'Fraud': '#d62728'},
    title='Failed Authentication Attempts by Fraud Label',
    labels={'failed_auth_attempts': 'Failed Auth Attempts', 'Fraud': ''},
    points='outliers',
    template='plotly_white'
)
fig4.show()

In [35]:

def plot_distribution(col, color, title):
    data = df[col].dropna()
    mean_val   = data.mean()
    median_val = data.median()
    skewness   = data.skew()

    # KDE curve
    kde = gaussian_kde(data, bw_method=0.3)
    x_range = np.linspace(data.min(), data.quantile(0.99), 300)
    y_kde = kde(x_range)

    fig = go.Figure()

    # Histogram (normalized to density)
    fig.add_trace(go.Histogram(
        x=data,
        histnorm='probability density',
        nbinsx=80,
        name='Histogram',
        marker_color=color,
        opacity=0.5,
        xbins=dict(end=data.quantile(0.99))
    ))

    # KDE line
    fig.add_trace(go.Scatter(
        x=x_range, y=y_kde,
        mode='lines', name='KDE',
        line=dict(color=color, width=2.5)
    ))

    # Mean / Median vertical lines
    for val, label, dash, lcolor in [
        (mean_val,   f'Mean: {mean_val:,.2f}',   'dash',  'blue'),
        (median_val, f'Median: {median_val:,.2f}','dot',   'green'),
    ]:
        fig.add_vline(x=val, line_dash=dash, line_color=lcolor, line_width=2,
                      annotation_text=label, annotation_position='top right',
                      annotation_font_size=11)

    fig.update_layout(
        title=f'{title}<br><sup>Skewness: {skewness:.4f} — {"Right-skewed ▶" if skewness > 0 else "Left-skewed ◀" if skewness < 0 else "Symmetric"}</sup>',
        xaxis_title=col,
        yaxis_title='Density',
        legend=dict(orientation='h', y=-0.15),
        bargap=0.05,
        template='plotly_white'
    )
    fig.show()

plot_distribution('transaction_amount',    'steelblue', 'Distribution of Transaction Amount')
plot_distribution('amount_deviation_ratio','indianred', 'Distribution of Amount Deviation Ratio')

# Checking sqewness values for both features

In [36]:
num_cols = df.select_dtypes(include='number').columns.tolist()
corr = df[num_cols].corr().round(2)

fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Correlation Matrix',
    aspect='auto'
)
fig.update_layout(
    width=900, height=800,
    coloraxis_colorbar=dict(title='r'),
    xaxis=dict(tickangle=45)
)
fig.show()
# No multicollinearity issues detected, all correlations are below 0.8 in absolute value.

# Missing colummns
~24-25% missing (heavy): transaction_amount, velocity_last_1hr, account_age_days, failed_auth_attempts, amount_deviation_ratio, distance_from_home_km, cvv_match

~9-10% missing (moderate): transaction_hour, device_type, location_mismatch, previous_fraud_flags, merchant_category, card_type, ip_country_match, transaction_channel, is_weekend

In [37]:
df.isna().sum().sort_values(ascending=False)

cvv_match                    3675
transaction_amount           3668
amount_deviation_ratio       3625
account_age_days             3617
failed_auth_attempts         3615
velocity_last_1hr            3605
distance_from_home_km        3561
is_weekend                   1451
merchant_category            1443
device_type                  1427
previous_fraud_flags         1420
location_mismatch            1403
ip_country_match             1398
transaction_channel          1397
transaction_hour             1378
card_type                    1325
is_fraudulent_transaction       0
dtype: int64

In [38]:
# View all unique values for columns that are categorical to decide on encoding strategy later
catCols = ['card_type','merchant_category','transaction_channel','device_type']
for col in catCols:
    print(f"{col}: {df[col].nunique()} unique values")

card_type: 4 unique values
merchant_category: 8 unique values
transaction_channel: 4 unique values
device_type: 4 unique values


# Feature engineering

In [39]:
# Create a cvv_match_missing, ip_country_match_missing and a transaction_amount missing column to capture missingness patterns in the data, which can be informative for the model.
adjusted_df = df.copy()  # Create a copy of the original dataframe to work with
adjusted_df['cvv_match_missing'] = adjusted_df['cvv_match'].isna().astype(int)
adjusted_df['transaction_amount_missing'] = adjusted_df['transaction_amount'].isna().astype(int)
adjusted_df['ip_country_match_missing'] = adjusted_df['ip_country_match'].isna().astype(int)

In [40]:
adjusted_df.info()
adjusted_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   transaction_amount          11332 non-null  float64
 1   transaction_hour            13622 non-null  float64
 2   device_type                 13573 non-null  object 
 3   location_mismatch           13597 non-null  float64
 4   previous_fraud_flags        13580 non-null  float64
 5   velocity_last_1hr           11395 non-null  float64
 6   merchant_category           13557 non-null  object 
 7   card_type                   13675 non-null  object 
 8   account_age_days            11383 non-null  float64
 9   failed_auth_attempts        11385 non-null  float64
 10  ip_country_match            13602 non-null  float64
 11  amount_deviation_ratio      11375 non-null  float64
 12  transaction_channel         13603 non-null  object 
 13  is_weekend                  135

,transaction_amount,transaction_hour,device_type,location_mismatch,previous_fraud_flags,velocity_last_1hr,merchant_category,card_type,account_age_days,failed_auth_attempts,ip_country_match,amount_deviation_ratio,transaction_channel,is_weekend,distance_from_home_km,cvv_match,is_fraudulent_transaction,cvv_match_missing,transaction_amount_missing,ip_country_match_missing
0,NaN,9.0,desktop,0.0,0.0,0.0,travel,NaN,170.0,0.0,1.0,-0.2920,NaN,0.0,5.89,NaN,0,1,1,0
1,NaN,NaN,mobile,0.0,0.0,NaN,travel,credit,189.0,0.0,1.0,NaN,pos,1.0,7.63,NaN,0,1,1,0
2,172.55,15.0,desktop,0.0,0.0,NaN,groceries,debit,NaN,0.0,1.0,-0.7713,mobile_app,0.0,22.43,1.0,0,0,0,0
3,2729.47,21.0,desktop,0.0,0.0,9.0,gambling,prepaid,22.0,4.0,NaN,5.9249,online,0.0,202.49,NaN,1,1,0,1
4,232.76,NaN,mobile,1.0,1.0,0.0,entertainment,debit,557.0,1.0,1.0,-0.5409,online,0.0,NaN,1.0,0,0,0,0


# Missing Data Imputation Strategy

## Why Different Columns Need Different Strategies

Missing data is not random across this dataset — some columns are missing ~25% of values, others ~10%. The imputation method chosen for each column depends on three factors:
1. **Data type** (continuous vs binary vs categorical)
2. **Distribution shape** (skewed vs symmetric)
3. **Fraud relevance** (whether missingness itself is a signal)

---

## Strategy 1 — MICE (Multiple Imputation by Chained Equations)
**Columns:** `velocity_last_1hr`, `account_age_days`, `failed_auth_attempts`, `amount_deviation_ratio`, `previous_fraud_flags`, `distance_from_home_km`

These are continuous behavioural features that are interrelated — for example, a user with many `failed_auth_attempts` is likely to also have high `velocity_last_1hr`. MICE exploits these relationships by modelling each missing column as a function of all other columns, iterating until the estimates stabilise. This produces more realistic imputed values than a simple summary statistic, preserving the correlation structure that the fraud model will rely on.

---

## Strategy 2 — Median Imputation
**Columns:** `transaction_amount`, `cvv_match`, `ip_country_match`, `transaction_hour`

- **`transaction_amount`**: Heavily right-skewed (mean ~$20,898 vs median ~$126). The mean is pulled up by extreme outlier transactions (max ~$995K), making it a poor representative value. Median is robust to outliers and fills with a typical transaction value.
- **`cvv_match` & `ip_country_match`**: Near-binary columns (0 or 1) where the median is 1.0, reflecting that the vast majority of legitimate transactions pass both checks. Binary indicator columns were created first to preserve the fact that these values were missing.
- **`transaction_hour`**: Uniformly distributed across 0–23. The median (12) is a neutral midpoint that does not bias the distribution toward morning or evening transactions.

---

## Strategy 3 — Mode Imputation (Binary Columns)
**Columns:** `location_mismatch`, `is_weekend`

Both columns are binary (0 or 1) and heavily skewed toward 0 — over 90% of non-null values are 0 for `location_mismatch` and ~70% for `is_weekend`. The mode (most frequent value) is 0 in both cases, making it the statistically correct choice. Filling with 1 would artificially inflate the rate of location mismatches and weekend transactions, which are associated with higher fraud risk.

---

## Strategy 4 — Mode Imputation (Categorical Columns)
**Columns:** `device_type`, `card_type`, `merchant_category`, `transaction_channel`

These are low-cardinality categorical columns (4–8 unique values each). There is no numeric relationship to exploit, so mode (most frequent category) is the standard approach. It avoids introducing a new "unknown" category that could confuse downstream encoding, while keeping the distribution of categories close to what was observed in the non-missing data.

## Why Indicator Columns Were Created First 
For `cvv_match`, `ip_country_match`, and `transaction_amount`, binary indicator columns (`_missing`) were created **before** imputation. This is because the *absence* of these values may itself carry fraud signal — for example, a transaction where the CVV was never provided is behaviorally different from one where the CVV was provided and matched. Once imputed, this distinction is lost unless captured separately."


   

In [41]:


# Step 1 — MICE for continuous interrelated fraud-signal columns
mice_cols = [
    'velocity_last_1hr', 'account_age_days', 'failed_auth_attempts',
    'amount_deviation_ratio', 'previous_fraud_flags', 'distance_from_home_km'
]
mice = IterativeImputer(max_iter=10, random_state=42)
adjusted_df[mice_cols] = mice.fit_transform(adjusted_df[mice_cols])

# Step 2 — Median for transaction_amount (heavily right-skewed, indicator already created)
adjusted_df['transaction_amount'] = adjusted_df['transaction_amount'].fillna(adjusted_df['transaction_amount'].median())

# Step 3 — Median for binary fraud-signal columns (indicators already created)
adjusted_df['cvv_match']        = adjusted_df['cvv_match'].fillna(adjusted_df['cvv_match'].median())
adjusted_df['ip_country_match'] = adjusted_df['ip_country_match'].fillna(adjusted_df['ip_country_match'].median())

# Step 4 — Median for transaction_hour (uniform 0–23, median=12 is neutral)
adjusted_df['transaction_hour'] = adjusted_df['transaction_hour'].fillna(adjusted_df['transaction_hour'].median())

# Step 5 — Mode for low-variance binary columns (~90% are 0)
adjusted_df['location_mismatch'] = adjusted_df['location_mismatch'].fillna(adjusted_df['location_mismatch'].mode()[0])
adjusted_df['is_weekend']        = adjusted_df['is_weekend'].fillna(adjusted_df['is_weekend'].mode()[0])

# Step 6 — Mode for categorical columns (4–8 unique values each)
cat_cols = ['device_type', 'card_type', 'merchant_category', 'transaction_channel']
for col in cat_cols:
    adjusted_df[col] = adjusted_df[col].fillna(adjusted_df[col].mode()[0])

In [42]:
print("Remaining nulls after imputation:")
print(adjusted_df.isna().sum()[adjusted_df.isna().sum() > 0].to_string() or "None — all columns fully imputed ✓")
print(f"\nShape: {adjusted_df.shape}")
adjusted_df.info()

Remaining nulls after imputation:
Series([], )

Shape: (15000, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   transaction_amount          15000 non-null  float64
 1   transaction_hour            15000 non-null  float64
 2   device_type                 15000 non-null  object 
 3   location_mismatch           15000 non-null  float64
 4   previous_fraud_flags        15000 non-null  float64
 5   velocity_last_1hr           15000 non-null  float64
 6   merchant_category           15000 non-null  object 
 7   card_type                   15000 non-null  object 
 8   account_age_days            15000 non-null  float64
 9   failed_auth_attempts        15000 non-null  float64
 10  ip_country_match            15000 non-null  float64
 11  amount_deviation_ratio      15000 non-null  float64
 12  transaction_channel  

## EDA

In [43]:
num_cols = adjusted_df.select_dtypes(include='number').columns.tolist()
corr = adjusted_df[num_cols].corr().round(2)

fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Correlation Matrix',
    aspect='auto'
)
fig.update_layout(
    width=900, height=800,
    coloraxis_colorbar=dict(title='r'),
    xaxis=dict(tickangle=45)
)
fig.show()
# No multicollinearity issues detected, all correlations are below 0.8 in absolute value.

# Normalize the skewed values

In [44]:
# ── Log-Transform Skewed Features ────────────────────────────────────────────
# These features are right-skewed (confirmed in EDA). log1p(x) = log(x+1)
# handles zeros safely and pulls extreme outliers toward the centre.
# Applied to adjusted_df BEFORE splitting so train and test get the same transform.
# StandardScaler inside the LR pipeline then centres and scales the result.


non_negative_cols = [
    'transaction_amount',
    'distance_from_home_km',
    'velocity_last_1hr',
    'account_age_days',
]
for col in non_negative_cols:
    n_clipped = (adjusted_df[col] < 0).sum()
    if n_clipped > 0:
        print(f"Clipping {n_clipped} negative values in {col} to 0")
    adjusted_df[col] = adjusted_df[col].clip(lower=0)

# amount_deviation_ratio skipped — legitimately negative when spend < account average
skewed_cols = non_negative_cols

for col in skewed_cols:
    adjusted_df[col] = np.log1p(adjusted_df[col])
    print(f"log1p applied → {col}  |  skew after: {adjusted_df[col].skew():.3f}")

print("\nNaN check after transform:")
print(adjusted_df[skewed_cols].isna().sum())


Clipping 731 negative values in distance_from_home_km to 0
Clipping 2 negative values in velocity_last_1hr to 0
Clipping 133 negative values in account_age_days to 0
log1p applied → transaction_amount  |  skew after: 2.611
log1p applied → distance_from_home_km  |  skew after: 0.356
log1p applied → velocity_last_1hr  |  skew after: 0.619
log1p applied → account_age_days  |  skew after: -1.597

NaN check after transform:
transaction_amount       0
distance_from_home_km    0
velocity_last_1hr        0
account_age_days         0
dtype: int64


# Feature Encoding

## Why One-Hot Encoding?

Four columns remain as object dtype after imputation: `device_type`, `card_type`, `transaction_channel`, and `merchant_category`. All four are **nominal categorical** — their categories have no natural numeric order (e.g. "mobile" is not greater than "desktop"). Assigning arbitrary integers (label encoding) would imply a false ordering that could mislead models sensitive to magnitude, such as logistic regression, SVM, and KNN.

**One-hot encoding** converts each category into a separate binary column (0 or 1), making no assumption about ordering. Since the target model is undecided, this is the safe default that works correctly for all model types.

## drop_first=True — Avoiding the Dummy Variable Trap
When one-hot encoding a column with N categories, only N−1 columns are needed — the Nth can always be inferred when all others are 0. Including all N creates **perfect multicollinearity**, which causes problems in linear models. `drop_first=True` removes the reference category from each column.

## Boolean → Integer Conversion
`pd.get_dummies` produces `bool` dtype columns. These are converted to `int` (0/1) to ensure compatibility with all sklearn models and downstream numeric operations.


In [45]:
# Step 1 — One-hot encode nominal categorical columns (drop_first avoids dummy variable trap)
cat_cols = ['device_type', 'card_type', 'transaction_channel', 'merchant_category']
adjusted_df = pd.get_dummies(adjusted_df, columns=cat_cols, drop_first=True)

# Step 2 — Convert bool columns produced by get_dummies to int for model compatibility
bool_cols = adjusted_df.select_dtypes(include='bool').columns
adjusted_df[bool_cols] = adjusted_df[bool_cols].astype(int)



In [46]:
# Verification
print(f"Final shape: {adjusted_df.shape}")
print(f"Remaining object columns: {adjusted_df.select_dtypes(include='object').columns.tolist() or 'None ✓'}")
print(f"Nulls introduced: {adjusted_df.isna().sum().sum()}")
print(f"Target preserved: {'is_fraudulent_transaction' in adjusted_df.columns}")
adjusted_df.dtypes.value_counts()

Final shape: (15000, 32)
Remaining object columns: None ✓
Nulls introduced: 0
Target preserved: True


int64      20
float64    12
Name: count, dtype: int64

In [47]:
adjusted_df.amount_deviation_ratio.isna

<bound method Series.isna of 0         -0.292000
1        110.719300
2         -0.771300
3          5.924900
4         -0.540900
            ...    
14995      0.447600
14996      2.063600
14997      0.837600
14998    110.702547
14999    109.927804
Name: amount_deviation_ratio, Length: 15000, dtype: float64>

In [48]:
leak_check = adjusted_df.corr()['is_fraudulent_transaction'].abs().sort_values(ascending=False)
print(leak_check.head(10))


is_fraudulent_transaction    1.000000
failed_auth_attempts         0.805418
previous_fraud_flags         0.778446
velocity_last_1hr            0.649704
ip_country_match             0.639989
distance_from_home_km        0.603351
location_mismatch            0.592873
cvv_match                    0.574380
account_age_days             0.564146
device_type_unknown          0.371699
Name: is_fraudulent_transaction, dtype: float64


In [49]:
# Save a copy of the cleaned dataframe for modeling
adjusted_df.to_csv('fraud_transaction_dataset_cleaned.csv', index=False)

# Train Test Split

In [50]:
X = adjusted_df.drop(columns=['is_fraudulent_transaction'])
y = adjusted_df['is_fraudulent_transaction']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Model Selection & Hyperparameter Optimisation

## Why Bayesian Search?

Choosing the right hyperparameters for a machine learning model is not guesswork — it directly determines how well the model generalises to unseen data. There are three common approaches:

| Method | How it works | Problem |
|---|---|---|
| **Grid Search** | Tries every combination of hyperparameters | Computationally expensive — grows exponentially with each parameter added |
| **Random Search** | Tries random combinations | More efficient than grid search but still wastes trials on poor regions of the search space |
| **Bayesian Search** | Learns from each trial to intelligently pick the next set of hyperparameters | Finds good hyperparameters in far fewer trials |

**Bayesian Search** builds a probabilistic model (a surrogate model) of the relationship between hyperparameters and model performance. After each trial, it updates its belief about which regions of the search space are most promising and focuses future trials there. This means it converges on optimal hyperparameters faster — typically in 30–50 trials versus hundreds for grid search.

For our fraud detection dataset with three models to tune, this efficiency matters: Bayesian search explores all three models in a fraction of the time grid search would take.

---

## Why SMOTE Inside the Pipeline?

SMOTE (Synthetic Minority Oversampling Technique) generates synthetic fraud samples to balance the training data. It must live **inside the pipeline** rather than being applied to the full training set beforehand.

If SMOTE runs before cross-validation, synthetic samples generated from the full training set will have "seen" data from validation folds — this is **data leakage**. The model appears to perform better than it actually will in production.

By placing SMOTE inside the pipeline, it runs fresh on each training fold only, keeping validation folds completely unseen and untouched. This gives an honest estimate of real-world performance.

---

## Why These Three Algorithms?

The three models are chosen to cover a spectrum from simple to complex, allowing us to understand whether the fraud patterns in this dataset are linear or require more sophisticated decision boundaries.

### Logistic Regression — The Baseline
Logistic Regression is the simplest classification model. It learns a linear boundary between fraud and legitimate transactions by assigning a weight (coefficient) to each feature.

**Why include it:**
- Establishes a performance floor — if a complex model barely beats logistic regression, the added complexity is not worth it
- Fully interpretable — coefficients directly show which features push a transaction toward fraud
- Fast to train — provides a quick sanity check on the data and features

**Limitation:** Assumes a linear relationship between features and fraud probability. Real fraud patterns are rarely linear.

---

### Random Forest — The Robust Middle Ground
Random Forest builds hundreds of decision trees, each trained on a random subset of the data and features, then combines their predictions by majority vote.

**Why include it:**
- Handles the mix of binary indicators, continuous features, and one-hot encoded columns naturally
- Robust to the extreme outliers in `transaction_amount` and `distance_from_home_km` — trees split on thresholds, not magnitudes
- Built-in feature importance rankings help identify the most influential fraud signals
- Less prone to overfitting than a single decision tree due to the ensemble averaging

**Limitation:** Less interpretable than logistic regression and slower to train than gradient boosting.

---

### XGBoost — The High-Performance Benchmark
XGBoost (Extreme Gradient Boosting) builds trees sequentially, where each new tree corrects the errors of the previous ones. It is consistently the top performer on tabular fraud detection datasets in industry and competitions.

**Why include it:**
- Sequential error correction means it can capture subtle, complex fraud patterns that Random Forest misses
- `scale_pos_weight` parameter natively handles class imbalance by upweighting the fraud class during training
- Typically achieves the highest AUC and F1 on structured tabular data
- Fast relative to its performance level

**Limitation:** More hyperparameters to tune and less interpretable without additional tools like SHAP.

---

## Why PR-AUC is the Primary Metric

With only 8% of transactions being fraudulent, **accuracy is misleading** — a model that predicts "not fraud" for every transaction achieves 92% accuracy while catching zero fraud cases.

**PR-AUC (Precision-Recall Area Under Curve)** measures performance across all classification thresholds, focusing specifically on the minority (fraud) class. It answers the question: *across all possible thresholds, how well does the model balance catching fraud (recall) against avoiding false alarms (precision)?*

ROC-AUC is also reported but is less informative under heavy class imbalance — it can appear high even when the model performs poorly on the fraud class specifically. PR-AUC is the more honest metric here.


In [51]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search_configs = [
    (
        'LogisticRegression',
        Pipeline([
            ('smote', SMOTE(random_state=42)),
            ('scaler', StandardScaler()),
            ('model', LogisticRegression(max_iter=1000, random_state=42))
        ]),
        {
            'model__C': Real(0.01, 10, prior='log-uniform'),
            'model__class_weight': Categorical(['balanced', None])
        }
    ),
    (
        'RandomForest',
        Pipeline([
            ('smote', SMOTE(random_state=42)),
            ('model', RandomForestClassifier(random_state=42))
        ]),
        {
            'model__n_estimators': Integer(50, 500),
            'model__max_depth': Integer(3, 20),
            'model__min_samples_split': Integer(2, 20),
            'model__class_weight': Categorical(['balanced', None])
        }
    ),
    (
        'XGBoost',
        Pipeline([
            ('smote', SMOTE(random_state=42)),
            ('model', XGBClassifier(eval_metric='logloss', random_state=42))
        ]),
        {
            'model__n_estimators': Integer(50, 500),
            'model__max_depth': Integer(3, 10),
            'model__learning_rate': Real(0.01, 0.3, prior='log-uniform'),
            'model__subsample': Real(0.5, 1.0),
            'model__scale_pos_weight': Integer(1, 20)
        }
    ),
]

# Baysean Search

In [52]:
results = []

for name, pipeline, space in search_configs:
    print(f"\nRunning Bayesian search for {name}...")
    search = BayesSearchCV(
        estimator=pipeline,
        search_spaces=space,
        n_iter=30,
        scoring='roc_auc',
        cv=cv,
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    search.fit(X_train, y_train)
    results.append({
        'model': name,
        'best_score': search.best_score_,
        'best_params': search.best_params_,
        'fitted_search': search
    })
    print(f"{name} — CV AUC: {search.best_score_:.4f}")


Running Bayesian search for LogisticRegression...
LogisticRegression — CV AUC: 1.0000

Running Bayesian search for RandomForest...
RandomForest — CV AUC: 1.0000

Running Bayesian search for XGBoost...
XGBoost — CV AUC: 1.0000


In [53]:
# ── Pick Best Model & Evaluate on Test Set ────────────────────────────────────
best = max(results, key=lambda x: x['best_score'])
print(f"\nBest model: {best['model']} — CV AUC: {best['best_score']:.4f}")
print(f"Best params: {best['best_params']}")

best_model = best['fitted_search'].best_estimator_
y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))
print(f"ROC-AUC:  {roc_auc_score(y_test, y_proba):.4f}")
print(f"PR-AUC:   {average_precision_score(y_test, y_proba):.4f}")



Best model: LogisticRegression — CV AUC: 1.0000
Best params: OrderedDict({'model__C': 0.010253943538922507, 'model__class_weight': None})

Classification Report:
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00      2760
       Fraud       1.00      1.00      1.00       240

    accuracy                           1.00      3000
   macro avg       1.00      1.00      1.00      3000
weighted avg       1.00      1.00      1.00      3000

ROC-AUC:  1.0000
PR-AUC:   1.0000


In [54]:
def plot_feature_importance(name, model_step, feature_names):
    """Extract and plot feature importance for each model type."""

    # Logistic Regression — uses absolute coefficient values as importance
    if hasattr(model_step, 'coef_'):
        importances = np.abs(model_step.coef_[0])

    # Random Forest / XGBoost — native feature_importances_
    elif hasattr(model_step, 'feature_importances_'):
        importances = model_step.feature_importances_

    else:
        print(f"  Feature importance not available for {name}")
        return

    feat_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=True).tail(20)

    fig = go.Figure(go.Bar(
        x=feat_df['importance'],
        y=feat_df['feature'],
        orientation='h',
        marker_color='steelblue'
    ))
    fig.update_layout(
        title=f'{name} — Top 20 Feature Importances',
        xaxis_title='Importance',
        yaxis_title='Feature',
        height=600,
        template='plotly_white'
    )
    fig.show()


for result in results:
    name        = result['model']
    best_est    = result['fitted_search'].best_estimator_
    y_pred      = best_est.predict(X_test)
    y_proba     = best_est.predict_proba(X_test)[:, 1]
    model_step  = best_est.named_steps['model']

    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"  CV AUC:  {result['best_score']:.4f}")
    print(f"  ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
    print(f"  PR-AUC:  {average_precision_score(y_test, y_proba):.4f}")
    print(f"{'='*55}")
    print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))

    plot_feature_importance(name, model_step, X_train.columns.tolist())


  LogisticRegression
  CV AUC:  1.0000
  ROC-AUC: 1.0000
  PR-AUC:  1.0000
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00      2760
       Fraud       1.00      1.00      1.00       240

    accuracy                           1.00      3000
   macro avg       1.00      1.00      1.00      3000
weighted avg       1.00      1.00      1.00      3000




  RandomForest
  CV AUC:  1.0000
  ROC-AUC: 1.0000
  PR-AUC:  1.0000
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00      2760
       Fraud       1.00      1.00      1.00       240

    accuracy                           1.00      3000
   macro avg       1.00      1.00      1.00      3000
weighted avg       1.00      1.00      1.00      3000




  XGBoost
  CV AUC:  1.0000
  ROC-AUC: 1.0000
  PR-AUC:  1.0000
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00      2760
       Fraud       0.99      1.00      1.00       240

    accuracy                           1.00      3000
   macro avg       1.00      1.00      1.00      3000
weighted avg       1.00      1.00      1.00      3000



# Save the model

In [55]:

# Extract the best fitted LR pipeline from results
lr_result = next(r for r in results if r['model'] == 'LogisticRegression')
lr_pipeline = lr_result['fitted_search'].best_estimator_

# Save model and feature column list (needed for inference)
joblib.dump(lr_pipeline, 'fraud_lr_model.pkl')
joblib.dump(X_train.columns.tolist(), 'model_features.pkl')

print("Model saved: fraud_lr_model.pkl")
print("Features saved: model_features.pkl")


Model saved: fraud_lr_model.pkl
Features saved: model_features.pkl


In [56]:
# load the model
lr_pipeline = joblib.load('fraud_lr_model.pkl')


In [57]:
# ── Threshold Tuning (Logistic Regression) ────────────────────────────────────
y_proba_lr = lr_pipeline.predict_proba(X_test)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_lr)
f1_scores      = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx       = f1_scores.argmax()
best_threshold = thresholds[best_idx]

print(f"\nOptimal threshold: {best_threshold:.4f}")
print(f"Precision: {precisions[best_idx]:.3f}  Recall: {recalls[best_idx]:.3f}  F1: {f1_scores[best_idx]:.3f}")

y_pred_tuned = (y_proba_lr >= best_threshold).astype(int)
print("\nClassification Report (tuned threshold):")
print(classification_report(y_test, y_pred_tuned, target_names=['Legit', 'Fraud']))

fig_pr = go.Figure()
fig_pr.add_trace(go.Scatter(x=recalls, y=precisions, mode='lines',
                             name='PR Curve', line=dict(color='steelblue', width=2)))
fig_pr.add_trace(go.Scatter(x=[recalls[best_idx]], y=[precisions[best_idx]],
                             mode='markers', name=f'Optimal threshold ({best_threshold:.2f})',
                             marker=dict(color='red', size=10)))
fig_pr.update_layout(title='Precision-Recall Curve — Logistic Regression',
                     xaxis_title='Recall', yaxis_title='Precision', template='plotly_white')
fig_pr.show()

joblib.dump(best_threshold, 'model_threshold.pkl')
print(f"Threshold saved: model_threshold.pkl")


Optimal threshold: 0.4827
Precision: 1.000  Recall: 1.000  F1: 1.000

Classification Report (tuned threshold):
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00      2760
       Fraud       1.00      1.00      1.00       240

    accuracy                           1.00      3000
   macro avg       1.00      1.00      1.00      3000
weighted avg       1.00      1.00      1.00      3000



Threshold saved: model_threshold.pkl
